In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [5]:
df = pd.read_csv('/content/drive/MyDrive/clean_steam_dataset.csv')
print(f"Ucitano recenzija: {len(df)}")
print("\nPrvih 5 redova:")
print(df.head())
print("\nKolone:")
print(df.columns.tolist())
print("\nRaspodela review_score:")
print(df['review_score'].value_counts())

Ucitano recenzija: 43160

Prvih 5 redova:
             app_name  review_score  \
0            PAYDAY 2             1   
1  Grand Theft Auto V             1   
2              Arma 3             1   
3            PAYDAY 2             1   
4  Grand Theft Auto V            -1   

                                         review_text  
0  This game is great! I think anybody would love...  
1  I havent seen my friends or socialized since A...  
2  Well,the price is a little bit highwith all th...  
3  Shot a swat members helmet off, and it flew ar...  
4  I became Master of PC programming after Grand ...  

Kolone u fajlu:
['app_name', 'review_score', 'review_text']

Raspodela review_score:
review_score
-1    21585
 1    21575
Name: count, dtype: int64


In [6]:
#LSTM uzima 0 i 1 kao output oznake
df['label'] = df['review_score'].apply(lambda x: 1 if x == 1 else 0)

X = df['review_text'].values
y = df['label'].values

print(f"Broj recenzija: {len(X)}")
print(f"Procenat pozitivnih (1): {y.sum() / len(y) * 100:.1f}%")
print(f"Odnos klasa - 0: {(y == 0).sum()}, 1: {(y == 1).sum()}")

Broj recenzija: 43160
Procenat pozitivnih (1): 50.0%
Odnos klasa - 0: 21585, 1: 21575


In [7]:
#odvojiti 10% podataka za test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

#odvajamo validaciju i train
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=1/9, random_state=42, stratify=y_temp)

print(f"Trening: {len(X_train)} recenzija")
print(f"Validacija: {len(X_val)} recenzija")
print(f"Test: {len(X_test)} recenzija")
print(f"\nOdnos klasa u treningu - 0: {(y_train == 0).sum()}, 1: {(y_train == 1).sum()}")

Trening: 34528 recenzija
Validacija: 4316 recenzija
Test: 4316 recenzija

Odnos klasa u treningu - 0: 17268, 1: 17260


In [11]:
max_words = 20000

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

train_lengths = [len(seq) for seq in X_train_seq]
percentile_95 = np.percentile(train_lengths, 95)

max_len = int(percentile_95)  # = 275

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

In [10]:
model1 = Sequential()
model1.add(Embedding(input_dim=max_words, output_dim=64))
model1.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
model1.add(Dense(1, activation='sigmoid'))

model1.build(input_shape=(None, max_len))
model1.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model1.summary()

callbacks1 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/drive/MyDrive/checkpoints/model1_best.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

history1 = model1.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val),
                      epochs=20, batch_size=32, callbacks=callbacks1, verbose=1)

In [10]:
model2 = Sequential()
model2.add(Embedding(input_dim=max_words, output_dim=128))
model2.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model2.add(Dense(1, activation='sigmoid'))

model2.build(input_shape=(None, max_len))
model2.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model2.summary()

callbacks2 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/drive/MyDrive/checkpoints/model2_best.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

history2 = model2.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val),
                      epochs=20, batch_size=32, callbacks=callbacks2, verbose=1)

In [ ]:
model3 = Sequential()
model3.add(Embedding(input_dim=max_words, output_dim=128))
model3.add(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2))
model3.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model3.add(Dense(1, activation='sigmoid'))

model3.build(input_shape=(None, max_len))
model3.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
model3.summary()

callbacks3 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('/content/drive/MyDrive/checkpoints/model3_best.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

history3 = model3.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val),
                      epochs=20, batch_size=32, callbacks=callbacks3, verbose=1)

In [ ]:
model4 = Sequential()
model4.add(Embedding(input_dim=max_words, output_dim=128))
model4.add(Bidirectional(LSTM(128, recurrent_dropout=0.2, return_sequences=True)))
model4.add(Dropout(0.4))
model4.add(Bidirectional(LSTM(64, recurrent_dropout=0.2, return_sequences=False)))
model4.add(BatchNormalization())
model4.add(Dense(64, activation='relu'))
model4.add(Dropout(0.4))
model4.add(Dense(1, activation='sigmoid'))

model4.build(input_shape=(None, max_len))
model4.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model4.summary()

os.makedirs('/content/drive/MyDrive/checkpoints', exist_ok=True)

callbacks4 = [
    ModelCheckpoint('/content/drive/MyDrive/checkpoints/model4_best.keras', monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
]

history4 = model4.fit(X_train_pad, y_train, validation_data=(X_val_pad, y_val),
                      epochs=20, batch_size=128, callbacks=callbacks4, verbose=1)

In [ ]:
#evaluacija modela na test skupu

models = [model1, model2, model3, model4]
names = ['Model 1 (LSTM 64)', 'Model 2 (LSTM 128)', 'Model 3 (2xLSTM 128)', 'Model 4 (dvosmerni LSTM)']

print("Evaluacija modela na testnom skupu:")
print("-"*30)

results = []

for name, model in zip(names, models):
    y_prob = model.predict(X_test_pad, batch_size=256, verbose=0)
    y_pred = (y_prob >= 0.5).astype(int).flatten()

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append([name, acc, prec, rec, f1])

    print(f"\n{name}:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negativna', 'Pozitivna'], yticklabels=['Negativna', 'Pozitivna'])
    plt.title(f'Matrica konfuzije - {name}')
    plt.xlabel('Predvidjeno')
    plt.ylabel('Stvarno')
    plt.show()

print("Pregled rezultata:")
print("-"*30)
results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1'])
print(results_df.to_string(index=False))